# Calibration Exercise — Jaccard & Confusion Index

**Purpose:** Before reference data collection begins, this notebook checks whether interpreters
are applying the response design consistently. It combines each interpreter's individual CSV
into one dataset, then runs two complementary calibration diagnostics for three variables
(**LULC class**, **LULC subclass**, and **Change/No-Change**):

1. **Inter-Rater Agreement (Jaccard, per class)** — for each class, how much two interpreters'
   sets of samples assigned to that class overlap, with a separate heatmap per class rather than
   one averaged number.
2. **Confusion Index** — which classes interpreters most often disagree on, as a bar chart per
   variable, to target follow-up training.

**Why Jaccard specifically:** Jaccard gives no credit at all for agreeing on classes *other* than
the one being scored, so a dominant class (e.g. "No Change") cannot inflate scores for a rarer one
the way it can distort Kappa or even Gwet's AC1. This makes Jaccard the most sensitive of the
three companion notebooks to a problem specific to a single class -- and, unusually, it can be
broken down class by class rather than only reported as one overall number. A single averaged
Jaccard would defeat this purpose (it can look moderate while hiding a near-zero score on the one
class that matters most), so this notebook deliberately never averages it into a single figure.
For an overall single-number agreement score, see the companion `Calibration_Kappa.ipynb` (best
for balanced classes) or `Calibration_Gwet_AC1.ipynb` (best for imbalanced classes, but still one
blended number). All three share the same data-prep, Confusion Index, and export structure, so
results are directly comparable across notebooks.

**How to use this notebook:** Only edit **Section 1.1 — Settings** below. Everything after that
runs automatically off those settings. Run cells top to bottom.

**Outputs produced:**
- `operator_identity_mapping.csv` — background file linking anonymized `operator_XX` IDs to real
  interpreter names. Kept separate from all results so results can be shared without revealing
  who interpreted what.
- `calibration_results_jaccard.xlsx` — one workbook, one sheet per result table (Jaccard and
  Confusion Index, for each of the three variables), plus a summary sheet.


## 1. Configuration

**This is the only section you should need to edit.** Update the paths and column names below to
match your project, then run the whole notebook from top to bottom.


### 1.1 Settings

All project-specific values live here: file locations, column names, which classes count as what,
and display options. Nothing in the cells after this one needs to be touched -- they read these
settings automatically.


In [ ]:
# ============================================================
# REQUIRED -- update these for your project
# ============================================================

INPUT_FOLDER = "/home/sepal-user/Country_support/Benin/MIA/CSV/"    # folder with one CSV per interpreter
OUTPUT_FOLDER = "/home/sepal-user/Country_support/Benin/MIA/Calibration_Outputs/Jaccard/"    # this notebook's own output folder

RAW_ID_COLUMN = "id"    # column that uniquely identifies a sample/plot in each interpreter's CSV

LULC_COLUMN = "lu_t2"        # land-use/land-cover class column
SUBCLASS_COLUMN = "sub_t2"   # LULC subclass column
CHANGE_COLUMN = "CHnCH"      # change / no-change column

# Only used if CHANGE_COLUMN is not already in your data -- derives it as (CHANGE_FROM_LU_COLUMN == CHANGE_FROM_CLASS)
# and (CHANGE_TO_LU_COLUMN != CHANGE_FROM_CLASS) and (CHANGE_YEAR_COLUMN >= CHANGE_START_YEAR).
CHANGE_FROM_LU_COLUMN = "event[1]_from_lu"    # class the sample was originally labelled
CHANGE_TO_LU_COLUMN = "lu_t2_label"           # class the sample is labelled now
CHANGE_FROM_CLASS = "F"                       # the "from" class that counts as change (e.g. Forest)
CHANGE_YEAR_COLUMN = "event[1]_change_yr"     # year the change event was recorded
CHANGE_START_YEAR = 2014                      # only changes on/after this year count as change

# The three variables above, run as (display label, column) -- add/remove/reorder freely.
VARIABLES = [
    ("LULC", LULC_COLUMN),
    ("LULC_subclass", SUBCLASS_COLUMN),
    ("Change", CHANGE_COLUMN),
]

# ============================================================
# OPTIONAL -- only if your data has these; leave as None to skip
# ============================================================

RAW_OPERATOR_NAME_COLUMN = None     # column with each interpreter's real name, if your CSV has one; else identity comes from the filename
TIMESTAMP_COLUMN = None             # e.g. "interpretation_time" -- enables the vigilance/fatigue check in Section 4.1
CONFIDENCE_COLUMN = "uncert"        # e.g. "uncert" or "confidence" -- self-reported hard-to-interpret flag; set to None to skip Section 4.2
HARD_TO_INTERPRET_VALUE = False     # the value in CONFIDENCE_COLUMN that means "hard to interpret" (only used if CONFIDENCE_COLUMN is set)
UNCERTAINTY_REASON_COLUMN = "uncertainty_reason"    # free-text reason column, if your tool records one; else None

# Class colors for charts, keyed by class code and/or full name (case-insensitive) -- any class not listed here still gets a color automatically.
CLASS_COLOR_MAP = {
    "f": "#228B22", "forest": "#228B22",
    "g": "#FF8C00", "grassland": "#FF8C00",
    "c": "#FFD54F", "cropland": "#FFD54F",
    "s": "#FF0000", "settlement": "#FF0000", "settlements": "#FF0000",
    "w": "#1E90FF", "wetland": "#1E90FF",
    "o": "#A9A9A9", "otherland": "#A9A9A9", "other": "#A9A9A9",
}

# ============================================================
# ADVANCED -- internals; leave as-is unless you know you need to change them
# ============================================================

N_BOOTSTRAP = 1000     # bootstrap resamples for confidence intervals
ALPHA = 0.05           # 95% CI
RANDOM_SEED = 42       # for reproducibility


### 1.2 Install required packages (only if needed)

Everything below is **commented out by default**. This notebook uses pandas, numpy, matplotlib,
seaborn, scikit-learn, and openpyxl. If your environment already has these (SEPAL usually does),
skip this cell entirely. If you get a `ModuleNotFoundError` when running Section 1.3 below, come
back here, remove the `#` from the line, run it once, then continue.


In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn openpyxl --break-system-packages


### 1.3 Setup: imports & helper functions

**Imports** bring in the libraries this notebook uses (pandas for tables, matplotlib/seaborn for
charts, scikit-learn for statistics). **Helper functions** are small, reusable pieces of code
defined once here and used repeatedly later, so the logic for "draw a heatmap" or "rank operators
worst-first" only has to be written once. You don't need to understand *how* each one works to use
the notebook -- here's what each one *does*:

- `operator_sort_key` -- makes sure operators list in natural order (01, 02, ... 10, 11), not
  alphabetical order (which would wrongly put "10" before "2").
- `add_rank_column` -- adds a "needs attention first" ranking to a results table, and safely
  handles the rare case of a duplicate/backup CSV file being read in twice.
- `plot_operator_bar` -- draws the ranked, worst-first bar chart used throughout this notebook to
  show one score per operator.
- `class_color` -- looks up a fixed color for a class name (e.g. "Forest"), matching
  `CLASS_COLOR_MAP` above, with a sensible automatic fallback for classes you haven't configured.
- `sanitize_filename` -- turns a class name into something safe to use in a filename.
- `plot_agreement_heatmap` -- draws the main Jaccard heatmap, including the fixed 0-1
  color scale, the qualitative Poor/Moderate/High/Excellent bands next to the color bar, and
  readable text on both light and dark cells.
- `plot_operator_by_class_bars` -- draws the grouped, per-class bar chart used for Jaccard's
  per-operator breakdown (Section 2.2), since Jaccard is never collapsed into one number.

Run this cell once per session; nothing in it needs editing.


In [ ]:
# ---- Imports ----
import os
import warnings
from glob import glob
import itertools
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)
sns.set(style="whitegrid")


# ---- Helper functions (see the explanation above for what each one does) ----

def operator_sort_key(op):
    import re
    m = re.search(r"(\d+)\s*$", str(op))
    return int(m.group(1)) if m else str(op)


def short_operator_label(op):
    """'operator 03' -> '03' -- used on chart/heatmap tick labels to save horizontal space when
    there are many interpreters. The full form is kept in tables and exports."""
    import re
    m = re.search(r"(\d+)\s*$", str(op))
    return m.group(1).zfill(2) if m else str(op)


def add_rank_column(df, value_col, worst_is_high, rank_name="Rank (most concern first)"):
    if df.index.duplicated().any():
        dupes = sorted(set(df.index[df.index.duplicated()]))
        print(f"  WARNING: operator ID(s) {dupes} appeared more than once -- likely a duplicate "
              f"or backup CSV file for that interpreter in the input folder. Averaging the "
              f"duplicate rows so the notebook can continue; check Section 1.4's file list to "
              f"find and remove the extra file, then rerun.")
        df = df.groupby(level=0).mean(numeric_only=True)
    df = df.reindex(sorted(df.index, key=operator_sort_key))
    # ascending = NOT worst_is_high: for a metric where LOW is bad (e.g. Kappa, worst_is_high=False),
    # we want the smallest value to get rank 1 -- pandas .rank(ascending=True) does that. For a
    # metric where HIGH is bad (e.g. Confusion Index, worst_is_high=True), we want the largest
    # value to get rank 1 -- pandas .rank(ascending=False) does that. So the ascending argument is
    # always the opposite of worst_is_high.
    ranked = df[value_col].rank(ascending=not worst_is_high, method="min").astype(int)
    df.insert(0, rank_name, ranked)
    return df


def plot_operator_bar(df, value_col, title, save_path, worst_is_high, flag_fraction=0.25):
    """Horizontal bar chart, ranked worst-first. The worst `flag_fraction` of operators are
    highlighted in red."""
    plot_df = df.sort_values(value_col, ascending=not worst_is_high)
    n_flag = max(1, int(round(len(plot_df) * flag_fraction)))
    colors = ["#D62728" if i < n_flag else "#4C72B0" for i in range(len(plot_df))]

    plt.figure(figsize=(9, max(3, 0.45 * len(plot_df))))
    plt.barh([short_operator_label(i) for i in plot_df.index], plot_df[value_col], color=colors)
    plt.gca().invert_yaxis()  # highest-concern operator at the top
    plt.xlabel(value_col, fontsize=12, fontweight="bold")
    plt.ylabel("Operator", fontsize=12, fontweight="bold")
    plt.title(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def class_color(label, fallback_palette=itertools.cycle(sns.color_palette("tab10").as_hex())):
    key = str(label).strip().lower()
    if key in CLASS_COLOR_MAP:
        return CLASS_COLOR_MAP[key]
    color = next(fallback_palette)
    print(f"  Note: no configured color for class '{label}' -- using a default color. "
          f"Add it to CLASS_COLOR_MAP in Section 1.1 to fix its color permanently.")
    return color


def sanitize_filename(text):
    return "".join(c if c.isalnum() else "_" for c in str(text))


# Interpretation bands shown next to the Jaccard color bar. Jaccard measures raw overlap, not
# chance-corrected agreement, so it isn't read on the same scale as Kappa/Gwet's AC1 -- these
# bands are practical guidance, not a formal published standard the way Kappa's bands are.
AGREEMENT_BANDS = [(0.0, 0.30, "Poor"), (0.30, 0.60, "Moderate"), (0.60, 1.001, "Strong")]


def plot_agreement_heatmap(results_df, value_col, title, save_path, negative_color="#C0392B"):
    raters = sorted(set(results_df["Operator 1"]) | set(results_df["Operator 2"]), key=operator_sort_key)
    short_labels = [short_operator_label(r) for r in raters]
    mat = pd.DataFrame(np.nan, index=raters, columns=raters)
    labels = pd.DataFrame("", index=raters, columns=raters)

    for _, row in results_df.iterrows():
        r1, r2 = row["Operator 1"], row["Operator 2"]
        if raters.index(r1) > raters.index(r2):
            mat.loc[r1, r2] = row[value_col]
            labels.loc[r1, r2] = f"{row[value_col]:.2f}"
        else:
            mat.loc[r2, r1] = row[value_col]
            labels.loc[r2, r1] = f"{row[value_col]:.2f}"

    # Fixed 0-1 color scale on every heatmap, regardless of this dataset's actual range, so
    # heatmaps from different variables/projects are visually comparable at a glance. Values
    # below 0 (worse than chance agreement) are drawn in a distinct solid color rather than being
    # folded into the 0-1 scale, so they're visually obvious rather than looking identical to 0.
    upper_mask = np.triu(np.ones_like(mat, dtype=bool), k=0)
    negative_mask = upper_mask | mat.isna() | (mat >= 0)
    # Text color is readable against both light and dark cells: black on the lighter lower half
    # of the scale, white on the darker upper half, split at DARK_TEXT_THRESHOLD -- a fixed,
    # predictable rule (not full per-cell auto-contrast), matching the notebook's standard
    # protocol of consistent styling while staying legible at the very darkest values.
    DARK_TEXT_THRESHOLD = 0.6
    light_mask = upper_mask | mat.isna() | (mat < 0) | (mat >= DARK_TEXT_THRESHOLD)
    dark_mask = upper_mask | mat.isna() | (mat < 0) | (mat < DARK_TEXT_THRESHOLD)

    fig = plt.figure(figsize=(max(9, len(raters) + 3.5), max(7, len(raters))))
    ax = plt.gca()
    sns.heatmap(mat, annot=labels, fmt="", cmap="YlGnBu", mask=light_mask, vmin=0, vmax=1,
                square=True, linewidths=0.5, linecolor="white",
                annot_kws={"color": "black"}, ax=ax)
    sns.heatmap(mat, annot=labels, fmt="", cmap="YlGnBu", mask=dark_mask, vmin=0, vmax=1,
                square=True, linewidths=0.5, linecolor="white", cbar=False,
                annot_kws={"color": "white"}, ax=ax)
    if (~negative_mask).any().any():
        from matplotlib.colors import ListedColormap
        sns.heatmap(mat, annot=labels, fmt="", cmap=ListedColormap([negative_color]), mask=negative_mask,
                    square=True, linewidths=0.5, linecolor="white", cbar=False,
                    annot_kws={"color": "black"}, ax=ax)
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(facecolor=negative_color, label=f"{value_col} < 0 (worse than chance)")],
                  loc="lower left", bbox_to_anchor=(0, -0.32), frameon=False, fontsize=11)

    # Qualitative interpretation bands next to the color bar (Poor/Moderate/High/Excellent, or the
    # Jaccard equivalent -- see AGREEMENT_BANDS above). The metric name (e.g. "Kappa") is written
    # horizontally above the color bar, like a small title, rather than the default sideways/
    # rotated label -- easier to read without tilting your head. Both this title and the band
    # labels use the same single color (black) for visual consistency.
    cbar = ax.collections[0].colorbar
    cbar.ax.yaxis.set_tick_params(pad=8)
    cbar.ax.set_title(value_col, fontsize=12, fontweight="bold", color="black", pad=12)
    for lo, hi, band_label in AGREEMENT_BANDS:
        mid = (lo + min(hi, 1.0)) / 2
        cbar.ax.text(3.2, mid, band_label, va="center", ha="left", fontsize=10, color="black",
                      transform=cbar.ax.get_yaxis_transform())
    for lo, hi, _ in AGREEMENT_BANDS[1:]:
        cbar.ax.axhline(lo, color="white", linewidth=1.2)

    ax.set_xticklabels(short_labels, rotation=0)
    ax.set_yticklabels(short_labels, rotation=0)
    ax.set_xlabel("Operator", fontsize=12, fontweight="bold")
    ax.set_ylabel("Operator", fontsize=12, fontweight="bold")
    plt.title(title, fontsize=15, pad=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_operator_by_class_bars(long_df, title, save_path, operator_order):
    """Grouped bar chart: operator on x-axis, one bar per class (fixed class colors), for a
    per-class metric like Jaccard."""
    plot_df = long_df.copy()
    plot_df["operator"] = pd.Categorical(plot_df["operator"], categories=operator_order, ordered=True)
    plot_df = plot_df.sort_values("operator")
    plot_df["operator_short"] = plot_df["operator"].map(short_operator_label)
    short_order = [short_operator_label(o) for o in operator_order]
    classes_present = list(dict.fromkeys(plot_df["Class"]))
    palette = {c: class_color(c) for c in classes_present}

    plt.figure(figsize=(max(10, len(operator_order) * 1.8), 7))
    ax = sns.barplot(data=plot_df, x="operator_short", y="Jaccard", hue="Class",
                      order=short_order, hue_order=classes_present, palette=palette)
    for i in range(1, len(operator_order)):
        plt.axvline(x=i - 0.5, color="gray", linestyle="--", linewidth=0.7)
    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.set_xlabel("Operator", fontsize=12, fontweight="bold")
    ax.set_ylabel("Mean Jaccard", fontsize=12, fontweight="bold")
    plt.xticks(rotation=0)
    ax.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()


print(f"Outputs will be written to: {OUTPUT_FOLDER}")


### 1.4 Load, prepare & check data

Everything below runs automatically, in order: read every interpreter's file, verify they all
cover the same samples, derive the Change/No-Change column if it isn't already present, reshape
the data into the wide format the statistics need, and print a sanity-check summary -- all in one
place so you can review the whole data-prep story in one pass rather than jumping between cells.

This cell also prints and saves the operator identity mapping (who each anonymized ID really is)
and a per-operator class-count table for each variable, for your own records -- see the notes printed
inline for how these two are meant to be used differently (one is private, one is safe to share).


In [ ]:
# ---- Validate configuration ----
_required = {"INPUT_FOLDER": INPUT_FOLDER, "OUTPUT_FOLDER": OUTPUT_FOLDER, "RAW_ID_COLUMN": RAW_ID_COLUMN,
             "LULC_COLUMN": LULC_COLUMN, "SUBCLASS_COLUMN": SUBCLASS_COLUMN, "CHANGE_COLUMN": CHANGE_COLUMN}
_missing = [name for name, val in _required.items() if not val]
if _missing:
    raise ValueError(f"Section 1.1 is missing required setting(s): {_missing}. Fill these in before running the notebook.")
if not VARIABLES:
    raise ValueError("VARIABLES is empty in Section 1.1 -- add at least one (label, column) pair.")
if not os.path.isdir(INPUT_FOLDER):
    raise FileNotFoundError(f"INPUT_FOLDER does not exist: {INPUT_FOLDER}. Check Section 1.1.")

# ---- Combine interpreter files & assign anonymized IDs ----
csv_files = sorted(glob(os.path.join(INPUT_FOLDER, "*.csv")))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {INPUT_FOLDER}. Check INPUT_FOLDER in Section 1.1.")

print(f"=== Combine files ===")
print(f"Found {len(csv_files)} interpreter file(s):")
for f in csv_files:
    print(f"  - {os.path.basename(f)}")

raw_frames = {}          # anonymized_id -> dataframe
identity_records = []    # for the mapping file

for i, file in enumerate(csv_files, start=1):
    anon_id = f"operator {i:02d}"
    real_name = os.path.splitext(os.path.basename(file))[0]
    if RAW_OPERATOR_NAME_COLUMN:
        df = pd.read_csv(file)
        if RAW_OPERATOR_NAME_COLUMN in df.columns and df[RAW_OPERATOR_NAME_COLUMN].nunique() == 1:
            real_name = str(df[RAW_OPERATOR_NAME_COLUMN].iloc[0])
    else:
        df = pd.read_csv(file)

    if RAW_ID_COLUMN not in df.columns:
        raise ValueError(f"'{RAW_ID_COLUMN}' (RAW_ID_COLUMN) not found in {os.path.basename(file)}. Check Section 1.1.")

    df["operator"] = anon_id
    raw_frames[anon_id] = df
    identity_records.append({"operator": anon_id, "real_name": real_name, "source_file": os.path.basename(file)})

identity_df = pd.DataFrame(identity_records)
identity_path = os.path.join(OUTPUT_FOLDER, "operator_identity_mapping.csv")
identity_df.to_csv(identity_path, index=False)
print(f"\nIdentity mapping (who is who) -- saved to:\n  {identity_path}")
print(f"\n{identity_df.to_string(index=False)}")
print(f"\nNote: this table stays private to whoever runs this notebook -- every other table, chart, "
      f"and export uses only the anonymized operator ID, so results can be shared without singling anyone out.")

# ---- Check sample completeness across operators ----
# Each sample ID is expected to repeat once per operator; any sample missing from any operator's file is dropped from ALL operators.
print(f"\n=== Sample completeness ===")
row_counts = {op: len(df) for op, df in raw_frames.items()}
print("Rows per operator file:")
for op, n in row_counts.items():
    print(f"  {op}: {n}")

id_sets = {op: set(df[RAW_ID_COLUMN]) for op, df in raw_frames.items()}
common_ids = set.intersection(*id_sets.values())
print(f"\nSamples present in ALL operator files: {len(common_ids)}")
for op, ids in id_sets.items():
    missing = ids - common_ids
    if missing:
        print(f"  WARNING: {op} has {len(missing)} sample(s) not shared by all operators -- these will be excluded: {sorted(missing)[:10]}{'...' if len(missing) > 10 else ''}")
if len(common_ids) == 0:
    raise ValueError("No sample IDs are common to all operator files -- check RAW_ID_COLUMN and the input data.")

frames_aligned = [df[df[RAW_ID_COLUMN].isin(common_ids)].copy() for df in raw_frames.values()]
combined_df = pd.concat(frames_aligned, ignore_index=True, join="outer")
print(f"Combined dataset: {len(combined_df)} rows across {len(raw_frames)} operators and {len(common_ids)} common samples.")

# ---- Verify or derive the Change/No-Change column ----
print(f"\n=== Change / No-Change column ===")
if CHANGE_COLUMN in combined_df.columns:
    print(f"'{CHANGE_COLUMN}' already present in the data -- using as-is.")
else:
    print(f"'{CHANGE_COLUMN}' not found -- deriving it from Section 1.1's rules "
          f"({CHANGE_FROM_LU_COLUMN} == '{CHANGE_FROM_CLASS}' and {CHANGE_TO_LU_COLUMN} != '{CHANGE_FROM_CLASS}' "
          f"and {CHANGE_YEAR_COLUMN} >= {CHANGE_START_YEAR}).")
    combined_df[CHANGE_COLUMN] = np.where(
        (combined_df[CHANGE_FROM_LU_COLUMN] == CHANGE_FROM_CLASS) &
        (combined_df[CHANGE_TO_LU_COLUMN] != CHANGE_FROM_CLASS) &
        (combined_df[CHANGE_YEAR_COLUMN] >= CHANGE_START_YEAR),
        "Change", "No Change",
    )
print(combined_df[CHANGE_COLUMN].value_counts())

# ---- Confirm every configured variable's column made it into the data ----
missing_cols = [col for label, col in VARIABLES if col not in combined_df.columns]
if missing_cols:
    raise ValueError(f"Column(s) {missing_cols} from VARIABLES not found in the data -- check Section 1.1.")

# ---- Reshape to wide format ----
# Each variable is pivoted so rows = samples and columns = operators -- the shape the statistics below need.
def reshape_wide(df, value_col, id_col=RAW_ID_COLUMN, operator_col="operator"):
    wide = df.pivot_table(
        index=id_col, columns=operator_col, values=value_col,
        aggfunc=lambda x: x.unique()[0] if len(x.unique()) > 0 else None,
    ).reset_index()
    wide.columns.name = None
    return wide

wide_data = {}
print(f"\n=== Reshape to wide format ===")
for label, col in VARIABLES:
    wide_data[label] = reshape_wide(combined_df, col)
    print(f"{label}: {wide_data[label].shape[0]} samples x {wide_data[label].shape[1] - 1} operators")

# ---- Sanity checks ----
# A per-operator class-count table, to spot rare-class sparsity before it shows up as an undefined statistic later.
print(f"\n=== Sanity checks ===")
class_count_tables = {}   # label -> wide operator x class table

for label, col in VARIABLES:
    wide = wide_data[label]
    operator_cols = [c for c in wide.columns if c != RAW_ID_COLUMN]
    classes_seen = sorted(pd.unique(wide[operator_cols].values.ravel()))
    classes_seen = [c for c in classes_seen if pd.notna(c)]
    print(f"[{label}] classes observed: {classes_seen}")

    counts = pd.DataFrame({op: wide[op].value_counts() for op in operator_cols}).fillna(0).astype(int).T
    counts.index.name = "operator"
    class_count_tables[label] = counts

    print(f"Per-operator class counts [{label}]:")
    print(counts)
    print()

    class_counts_path = os.path.join(OUTPUT_FOLDER, f"per_operator_class_counts_{label}.csv")
    counts.to_csv(class_counts_path)
    print(f"Saved to: {class_counts_path}\n")


## 2. Calibration: Inter-Rater Agreement (Jaccard, per class)

For each class, measures how much two interpreters' sets of samples assigned to that class
actually overlap -- agreeing on a different class earns no credit here at all. Shared functions
used for all three variables, so the logic exists once.


In [ ]:
def jaccard_per_class(l1, l2, categories):
    """Per-class Jaccard (Intersection over Union): for each class, how much do the two
    interpreters' sets of samples assigned to that class overlap. Unlike Kappa/Gwet's AC1, this
    gives no credit at all for agreeing on classes OTHER than the one being scored -- so a
    dominant class (e.g. 'No Change') cannot inflate scores for other classes the way it can
    distort a single overall Kappa."""
    scores = {}
    for k in categories:
        set1, set2 = (l1 == k), (l2 == k)
        union = (set1 | set2).sum()
        scores[k] = (set1 & set2).sum() / union if union > 0 else np.nan
    return scores


def mean_jaccard(l1, l2, categories):
    vals = [v for v in jaccard_per_class(l1, l2, categories).values() if pd.notna(v)]
    return np.mean(vals) if vals else np.nan


def bootstrap_jaccard(l1, l2, categories, n_bootstrap, rng):
    n = len(l1)
    boot = np.empty(n_bootstrap)
    for b in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        boot[b] = mean_jaccard(l1[idx], l2[idx], categories)
    return boot


def pairwise_jaccard(wide_df, categories, id_col=RAW_ID_COLUMN, n_bootstrap=N_BOOTSTRAP, alpha=ALPHA, rng=rng):
    df = wide_df.dropna().copy()
    operator_cols = [c for c in df.columns if c != id_col]

    results, all_boot, per_class_records = [], [], []
    for op1, op2 in itertools.combinations(operator_cols, 2):
        l1, l2 = df[op1].values, df[op2].values
        valid = ~(pd.isna(l1) | pd.isna(l2))
        l1, l2 = l1[valid], l2[valid]
        if len(l1) == 0:
            continue

        class_scores = jaccard_per_class(l1, l2, categories)
        mean_j = mean_jaccard(l1, l2, categories)
        boot = bootstrap_jaccard(l1, l2, categories, n_bootstrap, rng)
        ci_lo, ci_hi = np.nanpercentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])

        results.append({
            "Operator 1": op1, "Operator 2": op2,
            "Mean Jaccard": round(mean_j, 3) if pd.notna(mean_j) else np.nan,
            "CI Lower": round(ci_lo, 3), "CI Upper": round(ci_hi, 3),
        })
        all_boot.extend(boot)
        for k, v in class_scores.items():
            per_class_records.append({"Operator 1": op1, "Operator 2": op2, "Class": k, "Jaccard": v})

    return pd.DataFrame(results), np.array(all_boot), pd.DataFrame(per_class_records)


### 2.1 Pairwise Jaccard, per class -- one heatmap per class

Because a single averaged Jaccard number would hide exactly the kind of problem this metric is
meant to catch, a separate heatmap is produced **for every class**, rather than one averaged
heatmap per variable. A printed summary table lists every class's average overlap, sorted
lowest-first, so you know which class's heatmap to check first.


In [ ]:
jaccard_class_results = {}  # label -> long-format per-pair-per-class Jaccard (Operator 1/2, Class, Jaccard)
jaccard_class_detail = {}   # label -> per-class Jaccard, averaged across all operator pairs

for label, col in VARIABLES:
    print(f"\n=== Jaccard: {label} ===")
    wide = wide_data[label]
    operator_cols = [c for c in wide.columns if c != RAW_ID_COLUMN]
    categories = sorted(pd.unique(wide[operator_cols].values.ravel()))
    categories = [c for c in categories if pd.notna(c)]

    _, _, class_df = pairwise_jaccard(wide, categories)
    jaccard_class_results[label] = class_df

    if class_df.empty:
        print(f"  No valid pairwise comparisons for {label} -- skipping.")
        continue

    class_summary = class_df.groupby("Class")["Jaccard"].mean().round(3).sort_values()
    jaccard_class_detail[label] = class_summary
    print(f"  Per-class Jaccard, averaged across all operator pairs (lowest overlap first):")
    print(class_summary)

    for cls in categories:
        cls_df = class_df[class_df["Class"] == cls][["Operator 1", "Operator 2", "Jaccard"]]
        if cls_df.empty or cls_df["Jaccard"].isna().all():
            continue
        plot_path = os.path.join(OUTPUT_FOLDER, f"jaccard_heatmap_{label}_{sanitize_filename(cls)}.png")
        plot_agreement_heatmap(cls_df, "Jaccard", f"Inter-Rater Agreement (Jaccard) — {label}: {cls}", plot_path)

print("\nHow to read this: the summary table above lists each class's Jaccard overlap, "
      "lowest first -- a low score means operators rarely agree on that specific class, "
      "even if their overall accuracy looks fine. Next step: open the heatmap for the "
      "lowest-scoring class(es) first and check whether one operator or the whole group "
      "is driving the low score.")


### 2.2 Per-operator agreement summary, per class

Same idea as 2.1, but kept per class rather than averaged into one number: for each operator,
their mean Jaccard *for each class*, across all their pairings. The table is operator x class; the
chart groups bars by operator with one color per class (same fixed colors as the Confusion Index
charts in Section 3), so a specific operator's weak class jumps out rather than being folded into
one overall number.


In [ ]:
operator_agreement_jaccard = {}   # label -> operator x class mean-Jaccard table

for label, _ in VARIABLES:
    class_df = jaccard_class_results.get(label)
    if class_df is None or class_df.empty:
        continue

    long = pd.concat([
        class_df[["Operator 1", "Class", "Jaccard"]].rename(columns={"Operator 1": "operator"}),
        class_df[["Operator 2", "Class", "Jaccard"]].rename(columns={"Operator 2": "operator"}),
    ])
    per_op_class = long.groupby(["operator", "Class"])["Jaccard"].mean().round(3)
    per_op_table = per_op_class.unstack("Class")

    # Defensive: an operator ID appearing more than once here usually means a duplicate/backup CSV
    # in the input folder for that interpreter. Collapse rather than crash, but flag it clearly.
    if per_op_table.index.duplicated().any():
        dupes = sorted(set(per_op_table.index[per_op_table.index.duplicated()]))
        print(f"  WARNING: operator ID(s) {dupes} appeared more than once in this table -- likely "
              f"a duplicate or backup CSV file for that interpreter in {INPUT_FOLDER}. "
              f"Averaging the duplicate rows so the notebook can continue; check the file list "
              f"printed in Section 1.4 to find and remove the extra file, then rerun.")
        per_op_table = per_op_table.groupby(level=0).mean()

    per_op_table = per_op_table.reindex(sorted(per_op_table.index, key=operator_sort_key))
    operator_agreement_jaccard[label] = per_op_table

    print(f"\n[{label}] Per-operator mean Jaccard by class:")
    print(per_op_table)

    operator_order = list(per_op_table.index)
    bar_path = os.path.join(OUTPUT_FOLDER, f"jaccard_by_operator_{label}.png")
    plot_operator_by_class_bars(long, f"Mean Jaccard by Operator & Class — {label}", bar_path, operator_order)

print("\nHow to read this: this table is operator x class, so you can see which class "
      "is dragging down which operator, rather than one averaged score. Next step: when "
      "an operator is low on one class only, target training on that class rather than "
      "a general refresher.")


## 3. Confusion Index

Identifies which classes interpreters most often disagree on (pairwise, no ground truth needed —
appropriate here since calibration has no reference/"truth" labels). Shown as a bar chart per
variable rather than a heatmap, and includes a per-sample "how many unique labels did operators
give this sample" diagnostic — the same kind of check used to flag samples for group discussion.
When a variable has many classes (e.g. a detailed subclass legend), the chart automatically grows
wider and uses larger, bold, high-resolution text so it stays readable even when there are a lot
of bars to fit.


In [ ]:
def pairwise_confusion(wide_df, id_col=RAW_ID_COLUMN):
    df = wide_df.dropna().copy()
    operator_cols = [c for c in df.columns if c != id_col]
    for c in operator_cols:
        df[c] = df[c].astype(str)

    all_labels = sorted(pd.unique(df[operator_cols].values.ravel()))
    all_labels = [l for l in all_labels if l not in ("nan", "None")]

    confusion_mats = {}
    for c1, c2 in itertools.combinations(operator_cols, 2):
        valid = df[c1].notna() & df[c2].notna() & (df[c1] != "nan") & (df[c2] != "nan")
        y_true, y_pred = df.loc[valid, c1], df.loc[valid, c2]
        cm = confusion_matrix(y_true, y_pred, labels=all_labels).astype(float)
        total = cm.sum()
        if total == 0:
            continue
        conf_idx = cm.copy()
        np.fill_diagonal(conf_idx, 0)
        conf_idx /= total
        confusion_mats[(c1, c2)] = pd.DataFrame(conf_idx, index=all_labels, columns=all_labels)

    if not confusion_mats:
        return None, all_labels, None

    avg_conf = sum(confusion_mats.values()) / len(confusion_mats)
    return avg_conf, all_labels, confusion_mats


def top_confused_classes(avg_conf, all_labels, top_n=3):
    records = []
    for cls in all_labels:
        row = avg_conf.loc[cls].drop(cls).sort_values(ascending=False).head(top_n)
        for confused_class, value in row.items():
            records.append({"Base Class": cls, "Confused With": confused_class, "Confusion Index (%)": round(value * 100, 2)})
    return pd.DataFrame(records)


def plot_confusion_bars(grouped_df, title, save_path, base_class_order):
    grouped_df = grouped_df.copy()
    grouped_df["Base Class"] = pd.Categorical(grouped_df["Base Class"], categories=base_class_order, ordered=True)
    grouped_df = grouped_df.sort_values("Base Class")

    confused_categories = list(dict.fromkeys(grouped_df["Confused With"]))  # first-seen order
    palette = {cat: class_color(cat) for cat in confused_categories}

    # Wider and taller as the number of classes grows, so bars/labels don't get crushed together
    # for a long legend (e.g. a detailed subclass system).
    n_classes = len(base_class_order)
    fig_w = max(11, n_classes * 1.6)
    fig_h = max(7, 6 + n_classes * 0.08)
    fontsize_scale = 1.0 if n_classes <= 8 else max(0.75, 8 / n_classes * 1.4)

    plt.figure(figsize=(fig_w, fig_h))
    ax = sns.barplot(data=grouped_df, x="Base Class", y="Confusion Index (%)",
                      hue="Confused With", hue_order=confused_categories, palette=palette)

    for i in range(1, n_classes):
        plt.axvline(x=i - 0.5, color="gray", linestyle="--", linewidth=0.7)

    ax.set_title(title, fontsize=18, fontweight="bold")
    ax.set_xlabel("Base Class", fontsize=14 * fontsize_scale, fontweight="bold")
    ax.set_ylabel("Confusion Index (%)", fontsize=14 * fontsize_scale, fontweight="bold")
    plt.xticks(rotation=45, ha="right", fontsize=max(9, 12 * fontsize_scale))
    plt.yticks(fontsize=max(9, 12 * fontsize_scale))
    ax.legend(title="Confused With", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=max(8, 10 * fontsize_scale))
    plt.tight_layout()
    # Higher DPI here specifically, since this chart is the one most likely to have many classes
    # packed in and needs to stay sharp when downloaded/zoomed.
    plt.savefig(save_path, dpi=400, bbox_inches="tight")
    plt.show()


In [ ]:
confusion_results = {}       # label -> top-confused-classes dataframe
confusion_matrices = {}      # label -> averaged pairwise confusion matrices (for 3.2)
sample_agreement_results = {}  # label -> per-sample unique-label-count dataframe

for label, col in VARIABLES:
    print(f"\n=== Confusion Index: {label} ===")
    avg_conf, all_labels, pairwise_mats = pairwise_confusion(wide_data[label])

    if avg_conf is None:
        print(f"  No valid pairwise comparisons for {label} -- skipping.")
        continue

    if len(all_labels) <= 2:
        print(f"  Only {len(all_labels)} class(es) present ({all_labels}) -- a confusion-index "
              f"chart isn't informative for a binary variable (it's just the mirror of itself). "
              f"Skipping the chart.")
    else:
        top_df = top_confused_classes(avg_conf, all_labels, top_n=3)
        confusion_results[label] = top_df
        confusion_matrices[label] = pairwise_mats
        print(top_df.head(10))

        bar_path = os.path.join(OUTPUT_FOLDER, f"confusion_index_{label}.png")
        plot_confusion_bars(top_df, f"Confusion Index — {label}", bar_path, base_class_order=all_labels)

    operator_cols = [c for c in wide_data[label].columns if c != RAW_ID_COLUMN]
    sample_df = wide_data[label].copy()
    sample_df["Unique Labels"] = sample_df[operator_cols].nunique(axis=1)
    sample_agreement_results[label] = sample_df

    summary = sample_df["Unique Labels"].value_counts().sort_index()
    print(f"\n  Per-sample label agreement ({label}):")
    for n_labels, n_samples in summary.items():
        print(f"    {n_samples} sample(s) have {n_labels} unique label(s) across operators")

    sample_agreement_path = os.path.join(OUTPUT_FOLDER, f"sample_unique_labels_{label}.csv")
    sample_df.to_csv(sample_agreement_path, index=False)
    print(f"  Saved to: {sample_agreement_path}")

print("\nHow to read this: each bar shows how often a class was confused with another, "
      "as a % of all comparisons. Next step: for the classes with the tallest bars, "
      "review the response design definitions with the group, or add example plots "
      "for that class pair.")


### 3.2 Per-operator confusion profile

The chart above shows confusion *aggregated across all operators*. This breaks it down per
operator: for each interpreter, their personal average confusion rate (across all their
pairings), so an unusually high-confusion operator stands out individually rather than being
averaged away in the group total. Table in operator-ID order; chart ranked worst-first.


In [ ]:
operator_confusion = {}   # label -> per-operator mean confusion-index dataframe

for label, _ in VARIABLES:
    mats = confusion_matrices.get(label)
    if not mats:
        continue

    per_op_totals = {}
    for (op1, op2), mat in mats.items():
        off_diag_mean = (mat.values.sum() - np.trace(mat.values)) / (mat.shape[0] * (mat.shape[0] - 1))
        per_op_totals.setdefault(op1, []).append(off_diag_mean)
        per_op_totals.setdefault(op2, []).append(off_diag_mean)

    per_op_df = pd.DataFrame({
        "operator": list(per_op_totals.keys()),
        "Mean Confusion Index": [round(np.mean(v) * 100, 3) for v in per_op_totals.values()],
        "Pairings": [len(v) for v in per_op_totals.values()],
    }).set_index("operator")
    per_op_df = add_rank_column(per_op_df, "Mean Confusion Index", worst_is_high=True)

    operator_confusion[label] = per_op_df
    print(f"\n[{label}] Per-operator mean confusion index:")
    print(per_op_df)

    bar_path = os.path.join(OUTPUT_FOLDER, f"confusion_by_operator_{label}.png")
    plot_operator_bar(per_op_df, "Mean Confusion Index", f"Mean Confusion Index by Operator — {label}", bar_path, worst_is_high=True)

print("\nHow to read this: operators at the top of the chart (rank 1) confuse classes "
      "most often. Next step: cross-check them against Section 2.2 -- an operator low "
      "on both agreement and here needs follow-up before relying on their labels.")


### 3.3 Majority-vote consensus & operator deviation

For each sample, the majority label across all operators acts as a practical consensus reference
(not "truth" -- just what most interpreters agreed on). This flags: (a) samples with no clear
majority, i.e. high genuine confusion, worth a group discussion; and (b) which operators most
often land on the minority label, i.e. individually diverge from the group.


In [ ]:
majority_consensus = {}   # label -> per-sample consensus dataframe
ambiguous_samples = {}    # label -> dataframe of samples with no clear majority (for export)
operator_deviation = {}   # label -> per-operator deviation-from-majority dataframe

for label, _ in VARIABLES:
    wide = wide_data[label]
    operator_cols = [c for c in wide.columns if c != RAW_ID_COLUMN]
    df = wide.copy()

    def majority_label(row):
        counts = row[operator_cols].value_counts()
        if len(counts) == 0:
            return None, 0.0
        top_label = counts.index[0]
        is_tie = (counts == counts.iloc[0]).sum() > 1
        share = counts.iloc[0] / counts.sum()
        return (None if is_tie else top_label), share

    majority_info = df[operator_cols].apply(majority_label, axis=1)
    df["Majority Label"] = [m[0] for m in majority_info]
    df["Majority Share"] = [round(m[1], 3) for m in majority_info]
    majority_consensus[label] = df[[RAW_ID_COLUMN, "Majority Label", "Majority Share"] + operator_cols]

    tied = df[df["Majority Label"].isna()]
    ambiguous_samples[label] = tied[[RAW_ID_COLUMN] + operator_cols]
    tied_ids = tied[RAW_ID_COLUMN].tolist()
    print(f"\n[{label}] Samples with no clear majority (every label appears equally often -- "
          f"flag for group discussion): {len(tied_ids)}")
    if tied_ids:
        shown = tied_ids[:15]
        suffix = f" ... and {len(tied_ids) - 15} more" if len(tied_ids) > 15 else ""
        print(f"  Sample IDs: {shown}{suffix}")
        print(f"  (Full list with each operator's label, per sample, is in the "
              f"'Ambiguous_{label}' sheet of the results workbook.)")

    deviation_counts = {op: 0 for op in operator_cols}
    valid_rows = df[df["Majority Label"].notna()]
    for op in operator_cols:
        deviation_counts[op] = (valid_rows[op] != valid_rows["Majority Label"]).sum()

    dev_df = pd.DataFrame({
        "operator": list(deviation_counts.keys()),
        "Disagrees with Majority": list(deviation_counts.values()),
        "Total Comparable Samples": len(valid_rows),
    }).set_index("operator")
    dev_df["Disagreement Rate (%)"] = round(100 * dev_df["Disagrees with Majority"] / dev_df["Total Comparable Samples"], 2)
    dev_df = add_rank_column(dev_df, "Disagreement Rate (%)", worst_is_high=True)
    operator_deviation[label] = dev_df

    print(f"\n[{label}] Per-operator disagreement with majority:")
    print(dev_df)

    bar_path = os.path.join(OUTPUT_FOLDER, f"majority_deviation_by_operator_{label}.png")
    plot_operator_bar(dev_df, "Disagreement Rate (%)", f"Disagreement with Majority by Operator — {label}", bar_path, worst_is_high=True)

print("\nHow to read this: 'Disagreement Rate' is how often an operator's label "
      "differs from the group's majority label. Next step: discuss the 'Samples with "
      "no clear majority' list as a group -- these need agreement on the correct "
      "answer, not just individual training.")


## 4. Optional diagnostics

These two checks only run if the relevant column is configured **and** present in your data --
otherwise they're skipped with a short note, so the same notebook works cleanly whether or not a
given project's data includes them.


### 4.1 Vigilance / fatigue check (timestamp-based)

Checks whether disagreement with the majority label increases later in an operator's session --
evidence for (or against) limiting samples interpreted per day. Only runs if `TIMESTAMP_COLUMN`
is set in Section 1.1 and present in the data (only one of your two source tools currently
records this).


In [ ]:
if not TIMESTAMP_COLUMN or TIMESTAMP_COLUMN not in combined_df.columns:
    print(f"TIMESTAMP_COLUMN is not set or not present in this dataset -- skipping vigilance check. "
          f"(Set TIMESTAMP_COLUMN in Section 1.1 if this data source records interpretation timestamps.)")
else:
    ts_df = combined_df[[RAW_ID_COLUMN, "operator", TIMESTAMP_COLUMN]].dropna().copy()
    ts_df[TIMESTAMP_COLUMN] = pd.to_datetime(ts_df[TIMESTAMP_COLUMN])
    ts_df = ts_df.sort_values(["operator", TIMESTAMP_COLUMN])
    ts_df["session_position"] = ts_df.groupby("operator").cumcount()
    ts_df["session_length"] = ts_df.groupby("operator")["operator"].transform("count")
    ts_df["session_progress_bin"] = pd.qcut(
        ts_df["session_position"] / ts_df["session_length"], q=4,
        labels=["1st quarter", "2nd quarter", "3rd quarter", "4th quarter"],
    )

    primary_label = VARIABLES[0][0]
    consensus = majority_consensus.get(primary_label)
    if consensus is not None:
        merged = ts_df.merge(consensus[[RAW_ID_COLUMN, "Majority Label"]], on=RAW_ID_COLUMN, how="left")
        merged = merged.merge(
            combined_df[[RAW_ID_COLUMN, "operator", VARIABLES[0][1]]],
            on=[RAW_ID_COLUMN, "operator"], how="left",
        )
        merged["Disagrees with Majority"] = merged[VARIABLES[0][1]] != merged["Majority Label"]

        trend = merged.groupby("session_progress_bin", observed=True)["Disagrees with Majority"].mean() * 100
        print(f"Disagreement-with-majority rate by session progress ({primary_label}):")
        print(trend.round(2))
        if trend.iloc[-1] > trend.iloc[0]:
            print("\n  Disagreement rate is higher later in sessions -- possible fatigue/vigilance effect.")
        else:
            print("\n  No clear increase in disagreement later in sessions.")
    else:
        print("Majority-consensus data not available -- run Section 3.3 first.")


### 4.2 Repeated hard-to-interpret flags

A sample flagged as hard-to-interpret by **more than one** operator is a stronger signal than one
flagged by a single person -- these are the best candidates for group discussion or re-review.
Runs only if `CONFIDENCE_COLUMN` is set in Section 1.1 and present in the data. Also shown: which
**class** these samples fall into most often, and, if `UNCERTAINTY_REASON_COLUMN` is set, which
**reasons** come up most.


In [ ]:
if not CONFIDENCE_COLUMN or CONFIDENCE_COLUMN not in combined_df.columns:
    print(f"CONFIDENCE_COLUMN ('{CONFIDENCE_COLUMN}') is not set or not present in this dataset -- "
          f"skipping this check. Set CONFIDENCE_COLUMN in Section 1.1 if your data has one.")
else:
    # ---- How many different operators flagged each sample as hard-to-interpret? ----
    hard_flags = combined_df[combined_df[CONFIDENCE_COLUMN] == HARD_TO_INTERPRET_VALUE]
    sample_flag_counts = hard_flags.groupby(RAW_ID_COLUMN)["operator"].nunique().sort_values(ascending=False)
    sample_flag_counts.name = "Times Flagged Hard"

    repeated = sample_flag_counts[sample_flag_counts > 1]
    print(f"Samples flagged as hard-to-interpret by AT LEAST ONE operator: {len(sample_flag_counts)}")
    print(f"Samples flagged as hard-to-interpret by MORE THAN ONE operator: {len(repeated)}")

    if len(repeated) > 0:
        print(f"\n{repeated.to_string()}")
        repeated_ids = repeated.index.tolist()
        shown = repeated_ids[:20]
        suffix = f" ... and {len(repeated_ids) - 20} more" if len(repeated_ids) > 20 else ""
        print(f"\nSample IDs (flagged by more than one operator): {shown}{suffix}")

        repeated_path = os.path.join(OUTPUT_FOLDER, "repeatedly_hard_to_interpret_samples.csv")
        repeated.to_csv(repeated_path)
        print(f"Saved to: {repeated_path}")

        # ---- Which class are these repeatedly-flagged samples in? ----
        primary_label, primary_col = VARIABLES[0]
        repeated_class = combined_df[combined_df[RAW_ID_COLUMN].isin(repeated.index)][[RAW_ID_COLUMN, primary_col]].drop_duplicates(subset=RAW_ID_COLUMN)
        class_counts_repeated = repeated_class[primary_col].value_counts()
        print(f"\n[{primary_label}] Class breakdown of repeatedly-flagged samples:")
        print(class_counts_repeated)

        plt.figure(figsize=(max(8, len(class_counts_repeated) * 1.2), 6))
        colors = [class_color(c) for c in class_counts_repeated.index]
        plt.bar(class_counts_repeated.index.astype(str), class_counts_repeated.values, color=colors)
        plt.xlabel(primary_label, fontsize=12, fontweight="bold")
        plt.ylabel("Count of repeatedly-flagged samples", fontsize=12, fontweight="bold")
        plt.title(f"Repeatedly Hard-to-Interpret Samples by Class — {primary_label}", fontsize=14, fontweight="bold")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        class_bar_path = os.path.join(OUTPUT_FOLDER, f"repeatedly_hard_by_class_{primary_label}.png")
        plt.savefig(class_bar_path, dpi=300, bbox_inches="tight")
        plt.show()

        # ---- Reasons for uncertainty on these repeatedly-flagged samples, if recorded ----
        if UNCERTAINTY_REASON_COLUMN and UNCERTAINTY_REASON_COLUMN in combined_df.columns:
            reason_df = hard_flags[hard_flags[RAW_ID_COLUMN].isin(repeated.index)][[RAW_ID_COLUMN, UNCERTAINTY_REASON_COLUMN]].dropna()
            if len(reason_df) > 0:
                reason_counts = reason_df[UNCERTAINTY_REASON_COLUMN].value_counts()
                print(f"\nReasons given for uncertainty, on repeatedly-flagged samples:")
                print(reason_counts)

                plt.figure(figsize=(9, max(4, len(reason_counts) * 0.6)))
                plt.barh(reason_counts.index.astype(str), reason_counts.values, color="#FF8C00")
                plt.gca().invert_yaxis()
                plt.xlabel("Count", fontsize=12, fontweight="bold")
                plt.ylabel("Reason", fontsize=12, fontweight="bold")
                plt.title("Reasons for Uncertainty (repeatedly-flagged samples)", fontsize=14, fontweight="bold")
                plt.tight_layout()
                reason_bar_path = os.path.join(OUTPUT_FOLDER, "uncertainty_reasons_repeated.png")
                plt.savefig(reason_bar_path, dpi=300, bbox_inches="tight")
                plt.show()
            else:
                print(f"\nNo repeatedly-flagged samples have a value in '{UNCERTAINTY_REASON_COLUMN}'.")
        else:
            print(f"\nUNCERTAINTY_REASON_COLUMN is not set or not present in this dataset -- "
                  f"skipping the reasons breakdown. (Set it in Section 1.1 if your tool records why "
                  f"a sample was marked uncertain.)")
    else:
        print("\nNo sample was flagged as hard-to-interpret by more than one operator.")


## 5. Export consolidated results

All Jaccard, Confusion Index, and diagnostic tables are written into **one Excel workbook**, one
sheet per result table, so everything is in a single file to share. The operator identity mapping
(Section 1.4) is kept in its own separate file, since it's sensitive and shouldn't travel with
shared results.


In [ ]:
results_path = os.path.join(OUTPUT_FOLDER, "calibration_results_jaccard.xlsx")

with pd.ExcelWriter(results_path, engine="openpyxl") as writer:
    summary_rows = [{"Variable": label, "Operators": len([c for c in wide_data[label].columns if c != RAW_ID_COLUMN]),
                      "Samples": len(wide_data[label]),
                      "Worst-Class Jaccard": round(jaccard_class_detail[label].iloc[0], 3) if label in jaccard_class_detail and not jaccard_class_detail[label].empty else None,
                      "Worst Jaccard Class": jaccard_class_detail[label].index[0] if label in jaccard_class_detail and not jaccard_class_detail[label].empty else None}
                     for label, _ in VARIABLES]
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Summary", index=False)

    for label, _ in VARIABLES:
        if label in jaccard_class_results and not jaccard_class_results[label].empty:
            jaccard_class_results[label].to_excel(writer, sheet_name=f"Jaccard_{label}"[:31], index=False)
        if label in jaccard_class_detail:
            jaccard_class_detail[label].to_excel(writer, sheet_name=f"JaccardByClass_{label}"[:31])
        if label in operator_agreement_jaccard:
            operator_agreement_jaccard[label].to_excel(writer, sheet_name=f"JaccardByOp_{label}"[:31])
        if label in confusion_results:
            confusion_results[label].to_excel(writer, sheet_name=f"ConfIndex_{label}"[:31], index=False)
        if label in operator_confusion:
            operator_confusion[label].to_excel(writer, sheet_name=f"ConfByOp_{label}"[:31])
        if label in operator_deviation:
            operator_deviation[label].to_excel(writer, sheet_name=f"MajorityDev_{label}"[:31])
        if label in sample_agreement_results:
            sample_agreement_results[label].to_excel(writer, sheet_name=f"SampleAgree_{label}"[:31], index=False)
        if label in ambiguous_samples and not ambiguous_samples[label].empty:
            ambiguous_samples[label].to_excel(writer, sheet_name=f"Ambiguous_{label}"[:31], index=False)
        if label in class_count_tables:
            class_count_tables[label].to_excel(writer, sheet_name=f"ClassCounts_{label}"[:31])

print(f"Consolidated results saved to: {results_path}")


### 5.1 Outputs summary


In [ ]:
print("=== Outputs produced ===\n")
print(f"1. Operator identity mapping (background, sensitive):\n   {identity_path}\n")
print(f"2. Consolidated results workbook (Jaccard + Confusion Index + diagnostics, all variables):\n   {results_path}\n")
print("3. Heatmap and bar chart images:")
for label, _ in VARIABLES:
    if label in jaccard_class_detail:
        print(f"   - jaccard_heatmap_{label}_<class>.png (one per class)")
    if label in confusion_results:
        print(f"   - confusion_index_{label}.png")
print(f"\nAll saved under: {OUTPUT_FOLDER}")
